<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Práctico Final: MLOps Local con Monitoreo de Data Drift
## Fase 3 y Fase 5 — Servicio de Inferencia, Dashboard y Plan de Acción (Google Colab)
### Magíster en Ciencia de Datos — Tópicos en Data Science II
---

**Integrantes:** Gerardo Galán - Mabel Herrera

**Continúa de:** Fase 2 (`models/modelo_dengue.joblib`, ya entrenado) y Fase 4 (`reports/*.csv`, métricas de drift ya calculadas).

**Qué hace este notebook:**
1. Levanta el servicio de inferencia (FastAPI) de la Fase 3 dentro de esta misma sesión de Colab.
2. Lo alimenta con la producción simulada, exactamente como en una demo local.
3. Prueba en vivo el gatillo de reentrenamiento y el rollback de la Fase 5.
4. Expone el dashboard de Streamlit con una URL pública (vía `pyngrok`) para poder verlo desde el navegador.


---
### Nota sobre por qué esto corre distinto en Colab que en una máquina local

Localmente, el servicio, el dashboard y el script que los alimenta corren como **procesos separados** en distintas terminales. Colab es un solo notebook con un solo proceso, así que:

- El servicio FastAPI se levanta en un **hilo en segundo plano** dentro de esta misma sesión, en vez de una terminal aparte.
- El dashboard de Streamlit necesita una **URL pública** (`pyngrok`) para poder verse, porque no hay forma de abrir `localhost` directamente desde Colab.
- Esta sesión de Colab es **efímera**: todo lo que se genera (modelo reentrenado, base de datos, reportes) vive solo mientras dure esta ejecución. Por eso este notebook está pensado para correrse de principio a fin en una sola sesión continua — igual que será la demo en vivo de la defensa oral — y no para dejar resultados guardados entre sesiones distintas.


---
## Setup — Traer el repositorio a Colab e instalar dependencias


In [ ]:
# ── Bootstrap para Google Colab: trae el repositorio a este entorno ──
import os

REPO_URL = ""  # <-- si suben el proyecto a GitHub, peguen aqui la URL (ej. "https://github.com/usuario/dengue_mlops_drift_repo_1.git")
REPO_DIR = "/content/dengue_mlops_drift_repo_1"

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB and not os.path.exists(REPO_DIR):
    if REPO_URL:
        print("Clonando repositorio...")
        get_ipython().system('git clone -q "{}" "{}"'.format(REPO_URL, REPO_DIR))
    else:
        print("REPO_URL esta vacio. Selecciona el .zip del repositorio para subirlo:")
        from google.colab import files
        subido = files.upload()
        zip_name = list(subido.keys())[0]
        os.makedirs(REPO_DIR, exist_ok=True)
        get_ipython().system('unzip -q "{}" -d "{}"'.format(zip_name, REPO_DIR))
        contenido = os.listdir(REPO_DIR)
        if len(contenido) == 1 and os.path.isdir(os.path.join(REPO_DIR, contenido[0])):
            sub = os.path.join(REPO_DIR, contenido[0])
            for item in os.listdir(sub):
                os.rename(os.path.join(sub, item), os.path.join(REPO_DIR, item))
            os.rmdir(sub)

if IN_COLAB:
    os.chdir(REPO_DIR)  # aqui NO entramos a notebooks/ -- necesitamos service/ y dashboard/ como hermanos
    print("Directorio de trabajo:", os.getcwd())
else:
    print("No se detecto Colab -- se asume ejecucion local normal, sin cambios de directorio.")


In [ ]:
# ── Dependencias que Colab NO trae preinstaladas ──
# (pandas, numpy, scipy, scikit-learn, matplotlib, seaborn y joblib ya vienen listos en Colab)
get_ipython().system('pip install -q fastapi "uvicorn[standard]" httpx pydantic streamlit pyngrok')
print("Dependencias instaladas")


---
## Fase 3 — Levantar el servicio de inferencia dentro de Colab

Usamos el modelo que ya viene entrenado en `models/modelo_dengue.joblib` (Fase 2) — no hace falta reentrenar para esta demo. `service/app.py` es exactamente el mismo archivo que se usaría con `uvicorn app:app --port 8000` en una máquina local; aquí simplemente lo corremos en un hilo en vez de en una terminal aparte.


In [ ]:
# ── Levantar el servicio FastAPI en un hilo de fondo ──
import sys, threading, time
import requests

sys.path.insert(0, os.path.join(REPO_DIR, "service"))
sys.path.insert(0, REPO_DIR)
os.chdir(os.path.join(REPO_DIR, "service"))

from app import app as fastapi_app
import uvicorn

def _run_server():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(4)

r = requests.get("http://127.0.0.1:8000/health")
print("Estado del servicio:", r.status_code)
print(r.json())


---
## Alimentar el servicio con la producción simulada (demo real de Fase 3)

Corremos `service/simular_produccion.py` sin modificarlo — le manda al servicio, semana por semana, las 292 observaciones de `data/processed/produccion_simulada.csv`, exactamente como en la demo local.


In [ ]:
# ── Alimentar el servicio (puede tardar 1-2 minutos: son 292 peticiones reales) ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python simular_produccion.py --url http://127.0.0.1:8000')


In [ ]:
# ── Confirmar cuántas predicciones quedaron registradas ──
r = requests.get("http://127.0.0.1:8000/health")
print(r.json())


---
## Puente a la Fase 4 (métricas de drift)

El cálculo de PSI/KS por ventana y las alertas (`reports/drift_detalle.csv`, `reports/mae_por_ventana.csv`, `reports/alertas_por_ventana.csv`) ya está hecho y viene incluido en el repositorio — el dashboard de más abajo simplemente los lee, igual que en la versión local.

Si quieren **regenerarlos desde cero** en esta misma sesión de Colab (por ejemplo, porque acaban de reentrenar el modelo más abajo y quieren que el dashboard refleje el modelo nuevo), corran esta celda — ejecuta el notebook de Fase 4 completo de forma no interactiva, igual que sugiere el README para la opción de reentrenar desde cero:


In [ ]:
# ── OPCIONAL: regenerar reports/ ejecutando el notebook de Fase 4 completo ──
REGENERAR_REPORTS = False  # cambien a True si quieren recalcular el drift con el modelo actual

if REGENERAR_REPORTS:
    os.chdir(REPO_DIR)
    nb = "notebooks/practico_06_proyecto_final_fase4_drift_Gerardo_Galan_Mabel_Herrera.ipynb"
    get_ipython().system('jupyter nbconvert --to notebook --execute --inplace "{}"'.format(nb))
    print("reports/ regenerado con el modelo actual")
else:
    print("Se usan los reports/ ya incluidos en el repositorio (sin regenerar)")


---
## Fase 5 — Probar el gatillo de reentrenamiento y el rollback

Esto es exactamente `service/reentrenar.py`, sin modificar — el mismo script que se documenta en `PLAN_DE_ACCION.md`. Como esta copia del repositorio vive solo dentro de esta sesión de Colab, no hay ningún riesgo de tocar el modelo real del repositorio local: pueden correr esto las veces que quieran.


In [ ]:
# ── Ver el estado actual de alertas por ventana (Fase 4) ──
import pandas as pd
os.chdir(REPO_DIR)
pd.read_csv("reports/alertas_por_ventana.csv")


In [ ]:
# ── Reentrenar (ejemplo: San Juan, ventanas V2 y V3 -- ajustar segun lo que quieran demostrar) ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --city sj --ventanas V2 V3')


In [ ]:
# ── Ver el historial de versiones generado ──
import json
with open(os.path.join(REPO_DIR, "models", "historial_versiones.json")) as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


In [ ]:
# ── Probar el rollback ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --rollback')


---
## Dashboard de monitoreo (Streamlit) — exponerlo con `pyngrok`

**Antes de correr la celda siguiente:** creen una cuenta gratuita en [ngrok.com](https://dashboard.ngrok.com/signup), copien su *authtoken* desde el dashboard, y péguenlo abajo. Es un paso único — el token no cambia entre sesiones.


In [ ]:
# ── Configurar pyngrok (una sola vez) ──
from pyngrok import ngrok

NGROK_AUTHTOKEN = ""  # <-- pegar aqui el authtoken de https://dashboard.ngrok.com
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
else:
    print("Falta pegar el NGROK_AUTHTOKEN de https://dashboard.ngrok.com/get-started/your-authtoken")


In [ ]:
# ── Levantar el dashboard en segundo plano y exponerlo con un tunel publico ──
import subprocess, time

os.chdir(os.path.join(REPO_DIR, "dashboard"))
streamlit_proc = subprocess.Popen([
    "streamlit", "run", "dashboard.py",
    "--server.headless", "true", "--server.port", "8501",
])
time.sleep(6)

public_url = ngrok.connect(8501)
print("Dashboard disponible en:", public_url)


---
## Cierre y limitaciones de esta versión en Colab

- Todo lo generado en esta sesión (modelo reentrenado, base de datos de predicciones, historial de versiones) **se pierde al cerrar o reiniciar el entorno de ejecución** de Colab — es intencional (ver la nota de la Fase 5 más arriba); para conservarlo entre sesiones habría que montar Google Drive, que decidimos no hacer para mantener esto simple.
- El dashboard depende de que el túnel de `pyngrok` siga activo — si se cae la sesión de Colab, hay que volver a correr las dos últimas celdas.
- Todo el código que se ejecuta aquí (`service/app.py`, `service/simular_produccion.py`, `service/reentrenar.py`, `dashboard/dashboard.py`) es exactamente el mismo que corre localmente — no se duplicó ni se reescribió nada, solo se adaptó **cómo se lanza**.


In [ ]:
# ── Limpieza: detener el dashboard y cerrar el tunel ──
try:
    ngrok.disconnect(public_url)
except Exception:
    pass
streamlit_proc.terminate()
print("Dashboard y tunel detenidos. El servicio FastAPI del hilo de fondo se detiene solo al reiniciar el entorno de ejecucion.")
